# Module 2 — Chunking & Embeddings

**Duration:** ~90 minutes  
**Goal:** Understand how documents are prepared for a vector store — splitting into chunks and converting to embeddings.

---

## Why does chunking matter?

LLMs have a limited **context window**. You can't feed an entire DKV Belgium policy PDF into a single prompt.  
You need to:
1. Split documents into manageable pieces (**chunks**)
2. Represent each chunk as a vector (**embedding**)
3. At query time, retrieve only the *relevant* chunks

```
DKV Hospitalisation Policy
   │
   ├─ Chunk 1: "Article 1 — Définitions. Dans la présente police..."
   ├─ Chunk 2: "Article 2 — Garanties. L'assureur prend en charge..."
   ├─ Chunk 3: "Article 3 — Exclusions. Sont exclus de la garantie..."
   └─ Chunk N: ...
```

**The chunking strategy directly affects retrieval quality** — it's one of the most important tuning knobs in RAG.

---

## Chunking Trade-offs

| Chunk too small | Chunk too large |
|---|---|
| Loses context | Dilutes relevance signal |
| More noise in retrieval | Fewer chunks fit in context |
| Precise but fragmented answers | Broad but potentially off-topic |

**Sweet spot:** typically 600–1000 characters (≈ 100–200 words) with 10–15% overlap.

---

## We use LangChain text splitters

Instead of writing splitting logic from scratch, we use **`langchain-text-splitters`** — the industry standard for RAG.  
Three strategies to compare:

| Strategy | Splitter | When to use |
|---|---|---|
| A — Recursive | `RecursiveCharacterTextSplitter` | **Recommended default** — respects document structure |
| B — Fixed-size | `CharacterTextSplitter` | Baseline — simple but can cut mid-sentence |
| C — Token-aware | `SentenceTransformersTokenTextSplitter` | When you need to respect the model's token limit |

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    SentenceTransformersTokenTextSplitter,
)

# Load DKV Belgium documents
with open("../data/raw/corpus.json") as f:
    corpus = json.load(f)

print(f"Loaded {len(corpus)} documents")
for doc in corpus:
    print(f"  [{doc.get('language', '?').upper()}] {doc['title']}: {len(doc['content']):,} chars")

sample_doc = corpus[0]
sample_text = sample_doc["content"]
print(f"\nWorking with: '{sample_doc['title']}'")

---
## Part 1 — Chunking Strategies

### Strategy A: RecursiveCharacterTextSplitter (recommended default)

`RecursiveCharacterTextSplitter` tries a list of separators in order:  
`["\n\n", "\n", ". ", " ", ""]`  

It starts with the largest separator (paragraph breaks) and only falls back to smaller ones if a piece is still too large.  
This means a DKV article boundary or section header is **never cut** unless the section is genuinely too long.

> **Exercise 1.1** — Configure the splitter below.
>
> Use `chunk_size=800` (characters) and `chunk_overlap=100`.  
> Inspect a few chunks — notice how they tend to start at paragraph or sentence boundaries.

In [ ]:
# TODO: Create a RecursiveCharacterTextSplitter and split sample_text.
#
# splitter_recursive = RecursiveCharacterTextSplitter(
#     chunk_size=...,
#     chunk_overlap=...,
#     separators=["\n\n", "\n", ". ", " ", ""],
# )
# chunks_recursive = splitter_recursive.split_text(sample_text)

chunks_recursive = []  # replace with your implementation

print(f"Document: {len(sample_text):,} chars")
print(f"Chunks:   {len(chunks_recursive)}")
if chunks_recursive:
    sizes = [len(c) for c in chunks_recursive]
    print(f"Avg size: {sum(sizes)/len(sizes):.0f} chars  |  min: {min(sizes)}  max: {max(sizes)}")
    print(f"\n--- Chunk 1 ---")
    print(chunks_recursive[0])
    print(f"\n--- Chunk 2 ---")
    print(chunks_recursive[1] if len(chunks_recursive) > 1 else "(only one chunk)")

---
### Strategy B: CharacterTextSplitter (baseline)

The simplest approach: split purely on a separator character, no fallback logic.  
With `separator=" "` it produces roughly equal-size chunks but **can cut in the middle of a sentence**.

> **Exercise 1.2** — Configure a `CharacterTextSplitter`.
>
> Use the same `chunk_size=800` and `chunk_overlap=100`.  
> Compare the first chunk to Strategy A — does it start at a clean boundary?

In [ ]:
# TODO: Create a CharacterTextSplitter and split sample_text.
#
# splitter_fixed = CharacterTextSplitter(
#     separator=...,
#     chunk_size=...,
#     chunk_overlap=...,
# )
# chunks_fixed = splitter_fixed.split_text(sample_text)

chunks_fixed = []  # replace with your implementation

print(f"Chunks: {len(chunks_fixed)}")
if chunks_fixed:
    print(f"\n--- Chunk 1 (fixed) ---")
    print(chunks_fixed[0])
    print(f"\n--- Chunk 1 (recursive, for comparison) ---")
    print(chunks_recursive[0] if chunks_recursive else "(run Exercise 1.1 first)")

---
### Strategy C: SentenceTransformersTokenTextSplitter (token-aware)

**Why this matters for our workshop:**  
The multilingual embedding model (`paraphrase-multilingual-mpnet-base-v2`) has a **hard limit of 128 tokens**.  
At roughly 4 characters per token, an 800-character chunk is ~200 tokens — **well over the limit**.  
The model doesn't raise an error; it silently truncates. The last ~40% of the chunk is never embedded.

`SentenceTransformersTokenTextSplitter` uses the same tokenizer as the embedding model,  
so it knows exactly how many tokens each chunk uses.

> **Exercise 1.3** — Configure a `SentenceTransformersTokenTextSplitter`.
>
> Use `chunk_size=100` tokens and `chunk_overlap=10`.  
> After splitting, count the actual tokens in a chunk to verify it stays under 128.

In [ ]:
EMBED_MODEL = "paraphrase-multilingual-mpnet-base-v2"

# TODO: Create a SentenceTransformersTokenTextSplitter and split sample_text.
#
# splitter_tokens = SentenceTransformersTokenTextSplitter(
#     model_name=...,
#     chunk_size=...,
#     chunk_overlap=...,
# )
# chunks_tokens = splitter_tokens.split_text(sample_text)

chunks_tokens = []  # replace with your implementation

print(f"Chunks: {len(chunks_tokens)}")

if chunks_tokens:
    print(f"\n--- Chunk 1 ---")
    print(chunks_tokens[0])

    # Verify token count stays under the model's 128-token limit
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(EMBED_MODEL)
    tokens = model.tokenizer.encode(chunks_tokens[0])
    print(f"\nToken count: {len(tokens)} (model max: 128)")
    if len(tokens) <= 128:
        print("✓ Within limit — this chunk will be fully embedded.")
    else:
        print("✗ Over limit — the model will truncate this chunk!")

---
### Compare the three strategies

In [ ]:
def char_counts(chunks):
    return [len(c) for c in chunks]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
strategies = [
    ("A — Recursive\n(800 chars)", chunks_recursive),
    ("B — Fixed-size\n(800 chars)", chunks_fixed),
    ("C — Token-aware\n(100 tokens)", chunks_tokens),
]

for ax, (title, chunks) in zip(axes, strategies):
    if chunks:
        counts = char_counts(chunks)
        ax.hist(counts, bins=20, color="steelblue", edgecolor="white")
        ax.set_title(f"{title}\nn={len(chunks)}, avg={sum(counts)/len(counts):.0f} chars")
        ax.set_xlabel("Characters per chunk")
        ax.set_ylabel("Count")
    else:
        ax.text(0.5, 0.5, "Not implemented yet", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title)

plt.tight_layout()
plt.show()

print("\nDiscussion questions:")
print("1. Which strategy gives the most consistent chunk sizes? Why?")
print("2. Strategy A produces fewer, larger chunks than C — what are the retrieval trade-offs?")
print("3. For DKV Belgium docs (FR/NL/EN mixed), which strategy do you expect to work best?")

---
### Exercise 1.4b — Parameter sensitivity (your turn)

The parameters above were all given to you. Now pick your own and observe what changes.

| Setting | chunk_size | chunk_overlap | Expected effect |
|---|---|---|---|
| Small | 400 | 50 | More chunks, more precise but potentially fragmented |
| Default | 800 | 100 | Recommended starting point |
| Large | 1200 | 200 | Fewer chunks, more context per chunk |

Modify `YOUR_CHUNK_SIZE` and `YOUR_OVERLAP` below and re-run.
**Write down your preferred settings** — you will carry them into notebook 03,
re-index with them, and measure the retrieval quality difference directly.

In [ ]:
YOUR_CHUNK_SIZE = 800   # ← try 400 or 1200
YOUR_OVERLAP    = 100   # ← try 50 or 200

splitter_custom = RecursiveCharacterTextSplitter(
    chunk_size=YOUR_CHUNK_SIZE,
    chunk_overlap=YOUR_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks_custom = splitter_custom.split_text(sample_text)
sizes_custom = [len(c) for c in chunks_custom]

rows = [("Your config", YOUR_CHUNK_SIZE, YOUR_OVERLAP, len(chunks_custom), int(sum(sizes_custom)/len(sizes_custom)))]
if chunks_recursive:
    s = [len(c) for c in chunks_recursive]
    rows.append(("Default", 800, 100, len(chunks_recursive), int(sum(s)/len(s))))

print(f"{'Config':<15} {'chunk_size':>12} {'overlap':>10} {'n_chunks':>10} {'avg_chars':>10}")
print("─" * 62)
for label, cs, ov, nc, ac in rows:
    print(f"{label:<15} {cs:>12} {ov:>10} {nc:>10} {ac:>10}")

print(f"\nYour settings: chunk_size={YOUR_CHUNK_SIZE}, chunk_overlap={YOUR_OVERLAP}")
print("Carry these into notebook 03 to see the retrieval quality impact.")

---
## Part 2 — Embeddings

An **embedding** is a numerical vector that captures the *semantic meaning* of text.  
Similar texts produce similar vectors. We measure similarity using **cosine similarity**.

```
"What is insurance?"          → [0.12, -0.34, 0.87, ...] (384 dimensions)
"Insurance protects against risk" → [0.15, -0.31, 0.84, ...] (similar!)
"The weather is nice today"   → [-0.23, 0.61, -0.12, ...] (very different)
```

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the multilingual embedding model (768-dim, covers FR/NL/EN — pre-baked in the Docker image)
# This takes ~5 seconds on first run; subsequent calls use the cache.
print("Loading multilingual model...")
model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
print(f"Model loaded. Vector dimensions: {model.get_sentence_embedding_dimension()}")
print(f"Model max sequence length: {model.max_seq_length} tokens  ← this is the truncation limit")

In [ ]:
# Multilingual similarity — same concept in FR/NL/EN
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns

sentences = [
    "L'assureur prend en charge les frais d'hospitalisation.",        # FR
    "De verzekeraar dekt de ziekenhuiskosten.",                       # NL — same meaning
    "The insurer covers hospitalisation costs.",                       # EN — same meaning
    "Le remboursement est limité à 30 jours par année civile.",       # FR — different topic
    "Zijn de premies fiscaal aftrekbaar?",                            # NL — different topic
]

embeddings = model.encode(sentences)
sim_matrix = cosine_similarity(embeddings)

labels = [s[:45] + "..." for s in sentences]
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(sim_matrix, annot=True, fmt=".2f",
            xticklabels=labels, yticklabels=labels,
            cmap="coolwarm", vmin=0, vmax=1, ax=ax)
plt.title("Cosine Similarity — Multilingual (FR / NL / EN)")
plt.tight_layout()
plt.show()

print("\nObservation: The first three sentences say the same thing in different languages.")
print("Does the model recognise that? What does this mean for cross-lingual retrieval?")

---
### Exercise 1.4 — Batch embed all chunks

> **Task:** Use `RecursiveCharacterTextSplitter` to chunk the full corpus and embed everything.
>
> **Steps:**
> 1. Create a `RecursiveCharacterTextSplitter` with `chunk_size=800, chunk_overlap=100`
> 2. For each document, split its `content` and collect `{"text", "source", "id"}` dicts
> 3. Embed all chunk texts with `model.encode()` and record the time
>
> This is the **indexing cost** — you pay it once, then queries are fast.

In [ ]:
# TODO: Create the splitter
# splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100,
#                                           separators=["\n\n", "\n", ". ", " ", ""])

# TODO: Build all_chunks — list of {"text", "source", "id"} dicts
all_chunks = []

for doc in corpus:
    # doc_chunks = splitter.split_text(doc["content"])
    # for i, text in enumerate(doc_chunks):
    #     all_chunks.append({"text": text, "source": doc["title"], "id": f"{doc['title']}_{i}"})
    pass

print(f"Total chunks: {len(all_chunks)}")

# TODO: Extract texts and embed
# texts = [c["text"] for c in all_chunks]
# start = time.time()
# all_embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)
# elapsed = time.time() - start
# print(f"Embedded {len(texts)} chunks in {elapsed:.1f}s — shape: {all_embeddings.shape}")
# print(f"Throughput: {len(texts)/elapsed:.0f} chunks/sec")

---
## Reflection Questions

1. **Separator hierarchy:** Why is `"\n\n"` first in the list for `RecursiveCharacterTextSplitter`?  
   What would happen if you put `" "` first?

2. **Token limit:** You saw the model caps at 128 tokens. With `chunk_size=800` characters and ~4 chars/token,  
   roughly what fraction of each chunk gets silently dropped?

3. **Overlap purpose:** Set `chunk_overlap=0` and rerun Strategy A. What happens to chunks near an article boundary?  
   Why might a question about a boundary topic return worse results?

4. **Cross-lingual retrieval:** The heatmap showed FR/NL/EN sentences about the same concept cluster together.  
   Can you query in English and retrieve a matching French chunk? Try it in the next notebook.

---

## What we built

- [x] RecursiveCharacterTextSplitter — production-standard chunking with smart fallback
- [x] CharacterTextSplitter — simple baseline for comparison
- [x] SentenceTransformersTokenTextSplitter — token-aware chunking that respects model limits
- [x] Multilingual similarity heatmap (FR/NL/EN)
- [x] Batch embedding with throughput measurement

## Next: Module 2 — Vector Store & Retrieval

We have chunks and their embeddings. Now we need to **store** them efficiently  
and **search** them at query time. Open `03_vector_store_and_retrieval.ipynb` →